# Flights

Gets tomorrow's arrivals and departures for each airport from AeroDataBox (RapidAPI). Needs the `city_icao` table (filled by Gans_City) and `rapid_api_key` in `.env`.

In [15]:
import pandas as pd
from getpass import getpass
import requests
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv

### Setup

In [16]:
def configure_pandas_display():
    """Show full DataFrames without truncation."""

    pd.set_option('display.max_rows', None)      # show all rows
    pd.set_option('display.max_columns', None)   # show all columns
    pd.set_option('display.max_colwidth', None)  # don't cut off long cell text
    pd.set_option('display.width', None)         # use the full terminal/notebook width

In [17]:
def build_connection_string(schema="gans", host="127.0.0.1", user="root", port=3306):
    """Ask for the MySQL password and return a SQLAlchemy connection string."""

    # the name of the MySQL database to connect to
    # the server address — 127.0.0.1 means "this same computer" (localhost)
    # the MySQL username to log in with
    # the network port MySQL listens on (3306 is MySQL's default)

    password = getpass('Enter MySQL password: ')      # the password for that MySQL user

    # Build the full connection URL that SQLAlchemy/pymysql needs to connect:
    # format: mysql+pymysql://<user>:<password>@<host>:<port>/<schema>
    connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
    return connection_string

In [18]:
def load_api_key():
    """Read the RapidAPI key from the .env file and return it (None if missing)."""
    load_dotenv()  # reads .env into environment variables
    api_key = os.getenv("rapid_api_key")
    if api_key:
        print("rapid_api_key is set")
        return api_key
    else:
        print('rapid_api_key is not set')

### Flights
Load the airports from `city_icao`, then fetch tomorrow's flights. The API only allows 12 hours per request, so each airport is split into two calls.

In [19]:
def get_airport(connection_string):
    """Load city_id, icao and name of every airport from the database."""
    return pd.read_sql(
        'SELECT city_id, icao, name FROM city_icao',
        con=connection_string,
    )

In [20]:
def fetch_flights(icao, api_key):
    """Get tomorrow's departures and arrivals for one airport as a list of dicts."""
    tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
    times = [("00:00", "11:59"), ("12:00", "23:59")]   # API allows max 12 hours per request

    rows = []
    for start, end in times:
        from_local = f"{tomorrow}T{start}"
        to_local = f"{tomorrow}T{end}"
        response = requests.get(
            f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{from_local}/{to_local}",
            headers={"x-rapidapi-key": api_key, "x-rapidapi-host": "aerodatabox.p.rapidapi.com"},
            params={"withCancelled": "false", "withCodeshared": "true",
                    "withCargo": "false", "withPrivate": "false"},
        )
        if response.status_code == 204:
            print(f"{icao}: {from_local} -> {to_local}: no flights in this window")
            continue
        if response.status_code != 200:
            print(f"{icao}: {from_local} -> {to_local}: status code {response.status_code}")
            continue

        flight_json = response.json()
        for depart in flight_json.get("departures", []):
            rows.append({
                "airport_icao": icao,
                "flight_num": depart.get("number"),
                "direction": "departure",
                "origin_icao": icao,
                "destination_icao": depart["movement"]["airport"].get("icao"),
                "scheduled_time": depart["movement"].get("scheduledTime", {}).get("local"),
            })
        for arriv in flight_json.get("arrivals", []):
            rows.append({
                "airport_icao": icao,
                "flight_num": arriv.get("number"),
                "direction": "arrival",
                "origin_icao": arriv["movement"]["airport"].get("icao"),
                "destination_icao": icao,
                "scheduled_time": arriv["movement"].get("scheduledTime", {}).get("local"),
            })
    return rows

In [21]:
def collect_flights(airport_df, api_key):
    """Fetch flights for every airport and return one DataFrame."""
    records = []
    for icao in airport_df['icao'].unique():
        records.extend(fetch_flights(icao, api_key))

    flight_df = pd.DataFrame(records).drop_duplicates()
    # local times look like "2026-09-29 08:15+02:00" — keep the local clock time
    flight_df["scheduled_time"] = pd.to_datetime(flight_df["scheduled_time"].str[:16])
    return flight_df

In [22]:
def save_flights(flight_df, connection_string):
    """Append flights to the flight_status table."""
    return flight_df.to_sql(
        name='flight_status', con=connection_string,
        if_exists='append', index=False,
    )

### Run

In [23]:
def run_flight_pipeline():
    """Run the whole flow: load key, read airports, fetch flights, save."""
    configure_pandas_display()
    api_key = load_api_key()
    connection_string = build_connection_string()
    airport_df = get_airport(connection_string)
    flight_df = collect_flights(airport_df, api_key)
    rows_saved = save_flights(flight_df, connection_string)
    print(f"Saved {rows_saved} rows")
    return flight_df

In [24]:
flight_df = run_flight_pipeline()
flight_df.head()

rapid_api_key is set


EDDT: 2026-10-02T00:00 -> 2026-10-02T11:59: no flights in this window
EDDT: 2026-10-02T12:00 -> 2026-10-02T23:59: no flights in this window
Saved 2751 rows


,airport_icao,flight_num,direction,origin_icao,destination_icao,scheduled_time
0,EDDB,EW 8988,departure,EDDB,LDSP,2026-10-02 05:55:00
1,EDDB,FR 401,departure,EDDB,LIPZ,2026-10-02 05:55:00
2,EDDB,AM 6219,departure,EDDB,EHAM,2026-10-02 06:00:00
3,EDDB,UX 3758,departure,EDDB,EHAM,2026-10-02 06:00:00
4,EDDB,DL 9455,departure,EDDB,EHAM,2026-10-02 06:00:00
